# Ethene CASSCF data combiner

Combine every static-grid frame with a reproducible, non-repeating sample of dynamic frames.

In [1]:
from pathlib import Path
import random

REPOSITORY_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'data').is_dir())
STATIC_PATH = REPOSITORY_ROOT / 'data/A01_ethene/static/A01_ethene_grid_static_NEVPT2_FINAL.xyz'
DYNAMIC_PATH = REPOSITORY_ROOT / 'data/A01_ethene/dynamic/A01_ethene_dynamic_NEVPT2_sanity_defaults_filtered.xyz'
OUTPUT_PATH = REPOSITORY_ROOT / 'data/A01_ethene/combined/A01_ethene_grid_static_NEVPT2_FINAL_plus_2989_dynamic.xyz'
REMAINING_DYNAMIC_PATH = REPOSITORY_ROOT / 'data/A01_ethene/combined/A01_ethene_dynamic_NEVPT2_sanity_defaults_filtered_without_2989_sampled.xyz'
SAMPLE_SIZE = 2_989
RANDOM_SEED = 42


def read_xyz_frames(path: Path) -> list[list[bytes]]:
    """Read extended-XYZ records without altering their bytes."""
    frames = []
    with path.open('rb') as handle:
        while atom_count_line := handle.readline():
            if not atom_count_line.strip():
                continue
            atom_count = int(atom_count_line)
            frame = [atom_count_line]
            for _ in range(atom_count + 1):
                line = handle.readline()
                if not line:
                    raise ValueError(f'Incomplete frame in {path}')
                frame.append(line)
            frames.append(frame)
    return frames


static_frames = read_xyz_frames(STATIC_PATH)
dynamic_frames = read_xyz_frames(DYNAMIC_PATH)
if SAMPLE_SIZE > len(dynamic_frames):
    raise ValueError(f'Requested {SAMPLE_SIZE} dynamic frames, but only {len(dynamic_frames)} are available')

selected_indices = random.Random(RANDOM_SEED).sample(range(len(dynamic_frames)), SAMPLE_SIZE)
selected_dynamic_frames = [dynamic_frames[index] for index in selected_indices]
selected_index_set = set(selected_indices)
remaining_dynamic_frames = [frame for index, frame in enumerate(dynamic_frames) if index not in selected_index_set]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('wb') as handle:
    for frame in [*static_frames, *selected_dynamic_frames]:
        handle.writelines(frame)
with REMAINING_DYNAMIC_PATH.open('wb') as handle:
    for frame in remaining_dynamic_frames:
        handle.writelines(frame)

assert len(selected_indices) == len(set(selected_indices))
assert len(static_frames) + len(selected_dynamic_frames) == 6_720
assert len(selected_dynamic_frames) + len(remaining_dynamic_frames) == len(dynamic_frames)
print(f'Static frames: {len(static_frames)}')
print(f'Selected dynamic frames: {len(selected_dynamic_frames)} (seed={RANDOM_SEED})')
print(f'Output frames: {len(static_frames) + len(selected_dynamic_frames)}')
print(f'Wrote: {OUTPUT_PATH.relative_to(REPOSITORY_ROOT)}')
print(f'Remaining dynamic frames: {len(remaining_dynamic_frames)}')
print(f'Wrote: {REMAINING_DYNAMIC_PATH.relative_to(REPOSITORY_ROOT)}')


Static frames: 3731
Selected dynamic frames: 2989 (seed=42)
Output frames: 6720
Wrote: data/A01_ethene/combined/A01_ethene_grid_static_NEVPT2_FINAL_plus_2989_dynamic.xyz
Remaining dynamic frames: 35652
Wrote: data/A01_ethene/combined/A01_ethene_dynamic_NEVPT2_sanity_defaults_filtered_without_2989_sampled.xyz
